# How to read an Avro file with Python

Read an Avro file's schema, load it into pandas with Rugo, select fields including nested ones, read through a reader schema, and convert to Parquet to query with Opteryx.

This notebook accompanies [How to read an Avro file with Python](https://docs.opteryx.app/docs/guides/read-an-avro-file-with-python) on docs.opteryx.app.

> **Experimental:** Rugo's Avro reader arrives in the next `rugo` release. Until then, the `rugo.avro` import in this notebook will fail.

In [ ]:
%pip install -q rugo opteryx-core pandas pyarrow

## Get the data

`space_missions.avro`: 4,630 orbital launch attempts, with each rocket as a nested record.

In [ ]:
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/opteryx-core/main/testdata/avro/space_missions.avro", "space_missions.avro")

## Read the schema first

The schema is in the header, so nothing is decoded.

In [ ]:
from rugo.avro import read_avro, read_metadata

meta = read_metadata("space_missions.avro")
print(meta.codec)
for field in meta.schema["fields"]:
    print(field["name"], field["type"])

## Load an Avro file into pandas

Logical types are decoded; the nested `Rocket` record arrives as JSON text.

In [ ]:
import pyarrow

with read_avro("space_missions.avro") as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

df.dtypes

In [ ]:
df.head()

## Read only the fields you need

A dotted name reads one field out of a nested record.

In [ ]:
with read_avro(
    "space_missions.avro",
    columns=["Company", "Mission", "Rocket.Name", "Rocket.Status"],
) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

df.head()

## Filtering rows

Avro has no statistics, so there's no `predicates` argument. Filter each morsel as it arrives.

In [ ]:
import pandas as pd

parts = []
with read_avro("space_missions.avro", columns=["Company", "Mission", "Price"]) as reader:
    for morsel in reader:
        chunk = morsel.to_arrow().to_pandas()
        parts.append(chunk[(chunk["Company"] == "SpaceX") & (chunk["Price"] < 60)])

df = pd.concat(parts, ignore_index=True)
len(df)

## Read through a reader schema

Drop fields, add fields with defaults, and reshape nested records.

In [ ]:
reader_schema = {
    "type": "record",
    "name": "Launch",
    "fields": [
        {"name": "Mission", "type": "string"},
        {"name": "Price", "type": ["null", "double"], "default": None},
        {"name": "Source", "type": "string", "default": "space_missions"},
        {"name": "Rocket", "type": {"type": "record", "name": "Rocket", "fields": [
            {"name": "Name", "type": "string"},
            {"name": "Stages", "type": ["null", "int"], "default": None},
        ]}},
    ],
}

with read_avro("space_missions.avro", reader_schema=reader_schema) as reader:
    df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()

df.head()

## Query with SQL: convert to Parquet

Rugo reads the Avro and writes Parquet; Opteryx queries it, with `->>` reaching into the nested record.

In [ ]:
from rugo import parquet

with read_avro("space_missions.avro") as reader:
    with open("space_missions.parquet", "wb") as f:
        with parquet.open_parquet_writer(f.write) as writer:
            for morsel in reader:
                writer.write_row_group(morsel)

!rugo schema space_missions.parquet

In [ ]:
import opteryx

morsels = opteryx.session().execute_to_morsels("""
    SELECT Company, COUNT(*) AS launches
      FROM READ_PARQUET('space_missions.parquet')
     WHERE Rocket->>'Status' = 'Active'
     GROUP BY Company
     ORDER BY launches DESC
     LIMIT 5
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()